In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/jilmedavidesco@gmail.com/pipeline_project/1_code/1_setup/2_utilities

In [0]:
dbutils.widgets.text("catalog", "atlone_enterprise", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
# Leer tabla oro de la compañia padre
df_parent = spark.table(f"{catalog}.{gold_schema}.parent_dim_{data_source}")
df_parent.show(5)

# change year to string
df_parent = df_parent.withColumn("year", F.col("year").cast("string"))

# Crear tabla estadarizada "dim_gross_price"
df_parent.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{gold_schema}.3_dim_{data_source}")

In [0]:
df_parent.printSchema()

In [0]:
df_child = spark.table(f"{catalog}.{gold_schema}.child_dim_{data_source}")
df_child.show(5)

In [0]:
df_child = (
    df_child
    .withColumn("year", F.year("month"))
    # 0 = non-zero price, 1 = zero price  ➜ non-zero comes first
    .withColumn("is_zero", F.when(F.col("gross_price") == 0, 1).otherwise(0))
)

w = (
    Window
    .partitionBy("product_code", "year")
    .orderBy(F.col("is_zero"), F.col("month").desc())
)


df_child_price = (
    df_child
      .withColumn("rnk", F.row_number().over(w))
      .filter(F.col("rnk") == 1)
)


In [0]:
## Take required cols

df_child_price = df_child_price.select("product_code", "year", "gross_price").withColumnRenamed("gross_price", "price_inr").select("product_code", "price_inr", "year")

# change year to string
df_child_price = df_child_price.withColumn("year", F.col("year").cast("string"))

df_child_price.show(5)

In [0]:
df_child_price.printSchema()

In [0]:
delta_table = DeltaTable.forName(spark, "atlone_enterprise.gold.3_dim_gross_price")

# Merge con tabla estadarizada "dim_gross_price"
delta_table.alias("target").merge(
    source=df_child_price.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()